# 01 - Exploration: the temporal audit, the label and the paths that failed

**Question.** At the moment an order is approved, which orders will end with a negative review (score 1 or 2)?
The support team can follow up on 10% of orders, so the system must produce a ranked list, not a yes/no.

**What this notebook does.** It is the lab notebook: how the data behaves, when each column becomes known,
why the label, the split and the embargo were chosen, and which attempts were abandoned.
The final system is narrated in `02_system.ipynb`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from IPython.display import display
from early_warning import audit, config, features as F, viz
from early_warning.config import SEED          # the single seed of the whole project
np.random.seed(SEED)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)
pd.options.display.float_format = "{:,.4f}".format

from early_warning import exploration as X
from early_warning.pipeline import ensure_loaded
from early_warning.modeling import dataset as DS, baseline as B, evaluate as E, leakage as L
engine = ensure_loaded()
print("seed:", SEED)

seed: 42


## 1. The data at a glance
Row counts of the typed tables that the ingestion layer loaded into PostgreSQL (the model layer reads only from there).

In [2]:
display(X.table_counts(engine))

,table_name,rows
0,product_category_translation,71
1,sellers,3095
2,geo_zip,19010
3,products,32951
4,order_payments,103886
5,order_items,112650
6,order_reviews,99224
7,orders,99441
8,customers,99441
9,geolocation,1000163


## 2. The decision moment: order approval
The brief fixes the decision moment at order approval. A few orders have no approval timestamp.
We look at who they are before deciding what to do with them (decision D2 in `decisions.md`).

In [3]:
display(X.approval_lag_summary(engine))
display(X.never_approved_profile(engine))

,never_approved,median_lag_minutes,p95_lag_hours,p99_lag_hours
0,160,21.0000,48.5000,90.2000


,order_status,orders,with_review
0,canceled,141,138
1,delivered,14,14
2,created,5,3


Most never-approved orders are cancelled before payment cleared. The system is triggered *by* an approval,
so such orders are outside its population: they are excluded from training and evaluation. The prediction
path still scores them with an imputed approval time (purchase + the median lag shown above) and flags them.

## 3. The label and the orders without a review
An order has a negative label when its review score is 1 or 2. Orders that never received a review have an
*unknown* outcome - not a positive and not a negative. Dropping them is a decision with a side effect, so we measure it.

In [4]:
display(X.label_coverage(engine))
display(X.multi_review_orders(engine))

,order_status,orders,without_review,negative_rate_pct
0,delivered,96478,646,12.8000
1,shipped,1107,75,69.4000
2,canceled,625,20,76.7000
3,unavailable,609,14,84.7000
4,invoiced,314,5,81.9000
5,processing,301,6,92.5000
6,created,5,2,66.7000
7,approved,2,0,50.0000


,orders_with_several_reviews,reviews_disagree_on_label
0,547,96


Only a small share of orders lacks a review, so excluding them changes little. Orders that were cancelled or never
delivered have a much higher negative rate: those outcomes are decided *after* approval, which is exactly why
`order_status` is a trap (section 4). A few hundred orders carry several reviews; we use the first one that was answered
and document the disagreement count above.

## 4. Temporal audit
For every column we ask: *at the moment of approval, does this value already exist?*
First the empirical checks that back the claims (all queries run against the loaded data), then the audit table
that the code enforces.

In [5]:
display(audit.run_evidence(engine))

,check,what,value
0,E1,orders approved before purchase (must be 0),0.0000
1,E2,orders whose estimated delivery date is before the purchase date (must be 0),0.0000
2,E3,items whose shipping limit is before the purchase (must be 0),0.0000
3,E3b,"items whose shipping limit is before the approval (tolerated, tiny share)",126.0000
4,E4,"share of orders that have item rows, percent",99.2200
5,E7,share of orders that have at least one payment row,100.0000
6,E8,"negative-review rate of canceled orders, percent",76.5000
7,E8b,"negative-review rate of delivered orders, percent",12.8000
8,E9,"share of surveys created BEFORE the parcel was delivered, percent",8.6000
9,E10,"reviews answered before the order approval (data anomaly, tiny)",20.0000


In [6]:
display(audit.audit_table()[['column','available_at','available_at_approval','source_of_certainty']])

,column,available_at,available_at_approval,source_of_certainty
0,purchase_hour,purchase (customer places the order),yes,Timestamp is written when the order is placed; approval cannot precede it (E1).
1,purchase_dow,purchase (customer places the order),yes,Same as purchase_hour (E1).
2,approval_lag_hours,approval (decision moment),yes,Both ends exist at the decision moment by definition (E1). Imputed for never-approved orders.
3,promised_days_from_purchase,purchase (customer places the order),yes,The estimate is shown to the customer at checkout; it is never earlier than purchase (E2).
4,promised_days_from_approval,approval (decision moment),yes,"Estimate exists since purchase, approval is the decision moment (E2)."
5,ship_limit_slack_days,approval (decision moment),yes,Seller deadline is derived from the purchase: it is never earlier than purchase (E3).
6,n_items,purchase (customer places the order),yes,The basket is fixed when the order is placed (E4).
7,n_sellers,purchase (customer places the order),yes,Basket composition is fixed at purchase (E4).
8,n_products,purchase (customer places the order),yes,Basket composition is fixed at purchase (E4).
9,total_price,purchase (customer places the order),yes,Prices are shown at checkout (E4).


### Columns that look useful but were set aside

In [7]:
display(audit.excluded_table())

,column,available_at,why_excluded,why_tempting
0,orders.order_status,after approval (final status of a finished order),The table stores the FINAL status. 'canceled' and 'unavailable' are decided after approval.,Negative-review rate is 76% for canceled and 85% for unavailable orders against 13% for delivere...
1,orders.order_delivered_customer_date,after delivery,"Exists only once the parcel arrives, weeks after the decision. Its NULLs also encode status.",Delivery delay is the strongest single driver of a bad review.
2,orders.order_delivered_carrier_date,after the seller hands over the parcel,Seller handover happens after approval; its NULLs also encode status.,Handover delay predicts delivery delay.
3,order_reviews.review_score / comment fields,after the customer answers the survey,This is the label itself (or text written together with it).,Perfect correlation with the target.
4,order_reviews.review_creation_date,"survey send date, mostly after delivery (E9)",Only 8% of surveys are created before delivery; it is a post-approval timestamp that also proves...,Looks like an innocent order timestamp.
5,naive all-time seller negative rate,needs reviews that do not exist yet,A GROUP BY over all reviews of a seller includes future reviews AND the order's own review.,The most natural way to build a reputation feature; the classic leak the brief warns about.


### One sentence per feature: the hypothesis it tests

In [8]:
display(audit.hypothesis_table())

,feature,hypothesis,temporal_control
0,purchase_hour,Orders placed late at night wait longer for approval and handling.,"row-local, no history involved"
1,purchase_dow,Weekend orders are processed more slowly.,"row-local, no history involved"
2,approval_lag_hours,A slow payment approval signals a fragile order and an unhappy customer.,"row-local, no history involved"
3,promised_days_from_purchase,A long promise irritates customers; a short one is easier to break.,"row-local, no history involved"
4,promised_days_from_approval,The time left for the seller after approval drives lateness risk.,"row-local, no history involved"
5,ship_limit_slack_days,A tight handover deadline makes lateness more likely.,"row-local, no history involved"
6,n_items,Larger baskets have more ways to go wrong.,"row-local, no history involved"
7,n_sellers,Multi-seller orders arrive in pieces.,"row-local, no history involved"
8,n_products,Heterogeneous baskets are harder to fulfil.,"row-local, no history involved"
9,total_price,Expensive orders raise customer expectations.,"row-local, no history involved"


## 5. When does a review become *known*? (label embargo)
A review is created when the survey is sent but the score only exists when the customer answers. History features and
training labels must therefore use the *answer* timestamp. The lag between approval and answer also tells us how long a
training order's label stays unknown, which sets the embargo between train and test.

In [9]:
display(X.review_lag_quantiles(engine))
print("embargo used (days):", config.LABEL_EMBARGO_DAYS)

,q,days_after_approval
0,0.5000,13.0000
1,0.9000,26.6000
2,0.9500,31.7000
3,0.9900,45.5000
4,0.9990,167.8000


embargo used (days): 46


The 99th percentile is about 45 days, so the embargo is 46 days: no training order is closer to the boundary than the
time it takes (for 99% of orders) for its label to arrive. A purge on top removes the few remaining rows whose label
arrived after the boundary.

## 6. Where to cut in time?

In [10]:
frame = DS.build_modeling_frame(engine)
monthly = X.monthly_volume_and_rate(frame)
display(monthly)
viz.volume_and_rate(monthly, config.SPLIT_BOUNDARY)

,month,orders,negative_rate
0,2016-09,1,1.0000
1,2016-10,315,0.2825
2,2016-12,1,0.0000
3,2017-01,752,0.1543
4,2017-02,1752,0.1604
5,2017-03,2667,0.1429
6,2017-04,2357,0.1540
7,2017-05,3659,0.1279
8,2017-06,3225,0.1271
9,2017-07,3939,0.1224


<Figure size 750x350 with 2 Axes>

In [11]:
display(X.boundary_sensitivity(frame))

,boundary,train_rows,test_rows,train_negative_rate,test_negative_rate,test_share_of_labelled
0,2018-02-01,42849,46493,0.1455,0.1457,0.4720
1,2018-04-01,54753,32618,0.1477,0.1151,0.3310
2,2018-06-01,69118,18864,0.1604,0.1079,0.1910


The boundary is 2018-04-01. It keeps a full Black-Friday/Christmas season and the early-2018 delivery problems in training,
and leaves the largest part of the data that still has complete labels (about a third) for evaluation. Cutting at
2018-02 would test on a period that is still very close to the shock months; cutting at 2018-06 leaves too little to
evaluate month by month. Note that the negative rate is lower in the evaluation period: the world changed (see drift).

## 7. A rule the operations team could apply by hand (baseline)

In [12]:
split = DS.time_split(frame)
print(split.summary)
rule, table = B.choose_baseline(split.train, config.CAPACITY)
display(table)
y_test = split.test["is_negative"].astype(int).to_numpy()
print("chosen on the TRAINING period:", rule)
print("capture rate on the evaluation period:", round(E.capture_rate(y_test, B.rule_scores(split.test, rule)), 4))

{'boundary': '2018-04-01', 'embargo_days': 46, 'train_cutoff': '2018-02-14', 'orders_total': 99441, 'excluded_never_approved': 160, 'excluded_no_review': 763, 'train_rows': 54753, 'train_negative_rate': 0.1477, 'embargoed_rows': 11050, 'purged_rows': 97, 'test_rows': 32618, 'test_negative_rate': 0.1151, 'train_decision_range': ['2016-09-15', '2018-02-13'], 'test_decision_range': ['2018-04-01', '2018-09-03']}


,rule,description,train_capture_rate
0,worst_seller_reputation,Follow up the orders of sellers with the highest share of bad reviews so far.,0.1334
1,longest_promised_delivery,Follow up the orders with the longest promised delivery time after approval.,0.1254
2,farthest_shipping_distance,Follow up the orders travelling the longest distance.,0.1172


chosen on the TRAINING period: worst_seller_reputation
capture rate on the evaluation period: 0.1679


## 8. Hypotheses: what do the features say before any model?
Negative-review rate by bucket of three audited features, on labelled training orders.

In [13]:
for col in ["promised_days_from_approval", "s_neg_rate_known", "distance_km"]:
    display(X.rate_by_bucket(split.train, col))

,promised_days_from_approval_bucket,orders,negative_rate
0,"(-145.001, 19.0]",11064,0.1301
1,"(19.0, 23.0]",13267,0.1364
2,"(23.0, 26.0]",9369,0.1421
3,"(26.0, 31.0]",11039,0.1601
4,"(31.0, 154.0]",10014,0.1740


,s_neg_rate_known_bucket,orders,negative_rate
0,"(0.009899999999999999, 0.0833]",11885,0.1097
1,"(0.0833, 0.107]",10242,0.1218
2,"(0.107, 0.136]",10422,0.1355
3,"(0.136, 0.161]",10839,0.1621
4,"(0.161, 0.614]",10845,0.1777


,distance_km_bucket,orders,negative_rate
0,"(-0.001, 137.925]",10789,0.1203
1,"(137.925, 357.549]",10789,0.1455
2,"(357.549, 548.743]",10789,0.1334
3,"(548.743, 892.322]",10790,0.1437
4,"(892.322, 3387.36]",10788,0.1620


### The external signal (public holidays from a free API)
Hypothesis: if the promised delivery window contains public holidays, carriers and post offices are closed, deliveries
slip and customers complain. The signal links to an order by date only. First the raw relationship, then the honest test:
the same model with and without the feature, with a bootstrap interval for the difference on the evaluation period.

In [14]:
display(X.rate_by_bucket(split.train, "n_holidays_in_promise_window", q=6))
from early_warning.modeling.ablation import external_signal_ablation
ablation = external_signal_ablation(split)
display(pd.Series(ablation, name="value").to_frame())

,n_holidays_in_promise_window_bucket,orders,negative_rate
0,0.0000,30661,0.1492
1,1.0000,15877,0.1352
2,2.0000,6896,0.1614
3,3.0000,1306,0.1907
4,4.0000,10,0.5000
5,5.0000,3,0.0000


,value
validation_capture_with,0.2419
validation_capture_without,0.2413
test_capture_with,0.2366
test_capture_without,0.2358
test_diff_ci95,"[-0.005391554240240459, 0.005042388068319453]"


**Result: negative.** The interval of the difference contains zero, so the holiday feature adds nothing measurable beyond
the other features (the promised lead time already absorbs most of the calendar effect). The hypothesis was tested and is
reported as rejected rather than hidden; the feature stays in the audited set because it is harmless, but the decision does
not depend on it.

## 9. Paths that failed

### 9.1 A random split looks better - and is wrong
Randomly splitting orders lets the model learn from the future (seller reputation, seasonal shocks) and flatters it: the table shows the random split ahead of the temporal one, mostly in AUC and by a smaller margin in capture rate. The gap is moderate, but it is not real performance, so the temporal split is used.

In [15]:
display(L.random_vs_temporal(frame, split))

,split,capture_rate,roc_auc
0,random split (WRONG),0.2433,0.6671
1,temporal split (used),0.2366,0.6376


### 9.2 The naive all-time seller rate
The most natural reputation feature is a plain `GROUP BY seller` over all reviews. It contains the future *and the order's own
review*. The leakage table below prices it (and the other tempting columns); the full comparison is in `02_system.ipynb`.

In [16]:
leaky = L.load_leaky_columns(engine)
display(L.leakage_cost(split, leaky)[["model", "capture_rate", "roc_auc", "capture_rate_gain_vs_honest"]])

,model,capture_rate,roc_auc,capture_rate_gain_vs_honest
0,honest model (audited columns only),0.2366,0.6376,0.0000
1,"honest + post_approval_operations (status, delivery delay, handover)",0.4207,0.7546,0.1841
2,honest + naive_all_time_seller_negative_rate,0.2976,0.7178,0.0610
3,honest + review_comment_length,0.4042,0.7921,0.1676
4,CHEAT model (all leaky columns),0.5422,0.8809,0.3056


### 9.3 Feature query performance (development notes)
The first version computed the history windows and all joins inside one plain view. Measured during development
(not re-run here) it took several seconds against the 500 ms budget. Pre-computing the expensive event-stream windows and the
row-local roll-ups in materialised views brought the serving query under budget (`02_system.ipynb` shows the measured plan).
The cost moved into the refresh step, which is timed separately.